In [1]:
from datetime import datetime
import pandas as pd
import numpy as np

import requests
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

import urllib.parse
import json

import os
from tqdm import tqdm

In [2]:
import sys
from pathlib import Path

# Add ../helper to sys.path
helper_path = Path(__file__).resolve().parent.parent / "helper"
sys.path.insert(0, str(helper_path))

# Now import your modules 
from config_GAM2025 import gam_info
from security_config import api_key

import test_functions
import functions

In [3]:
# country
country_codes = pd.read_excel(f"helper/{gam_info['lookup_file']}", sheet_name='Country Code')
country_codes = country_codes.rename(columns={'ATI': 'geo_country'})[['geo_country', 'PlaceID']]

# week 
week_tester = pd.read_excel(f"helper/{gam_info['lookup_file']}", sheet_name='GAM Period')
#week_tester['w/c'] = pd.to_datetime(week_tester['w/c'])

# site info - with api query
site_info = pd.read_excel(f"helper/{gam_info['lookup_file']}", sheet_name='Site_API').drop(columns='no results')
site_info['Report No.'] = site_info['Report No.'].astype(str)
site_info = site_info[site_info['script'] == '0_site_1v1ps']


# service codes
service_codes = pd.read_excel(f"helper/{gam_info['lookup_file']}", sheet_name='Service Code')#[cols]
service_codes = service_codes.rename(columns={'ATI (Level 2 site)': 'site_level2'})[['site_level2', 'ServiceID']].dropna()


# ingestion 

## API Query

In [4]:
i = 0
for index, row in site_info.iterrows():
    
    api_query = row['API']
    report_no = row['Report No.']

    #print(convert_url_to_query(api_query, start, end))
    print(f"starting report no {report_no}")
    print(api_query)
    for jndex, row in week_tester.iterrows():
        week_number = row['Week Number']
        filename = f"../data/raw/site/piano_reports/{gam_info['file_timeinfo']}_reportNo{report_no}_weekNo{week_number}.csv"
        
        # Check if the file exists, if so, continue to the next iteration
        if os.path.exists(filename):
            continue
            
        print(f"... iteration {filename}")
        start = row['w/c'] # dtype object
        end = row['week_ending'] # dtype object
        
        # convert to api query 
        query = functions.convert_url_to_query(api_query, start, end)
        
        # run api query 
        temp = functions.api_call(query)
        
        temp['w/c'] = start
        temp['timestamp_queryRun'] = datetime.now().strftime('%y%m%d-%H%M')
        temp['API'] = api_query

        if temp.shape[0] == 0:
            temp = pd.DataFrame({
                'w/c': [start],
                'timestamp_queryRun': [datetime.now().strftime('%y%m%d-%H%M')],
                'api_query': [api_query]
            })
            
        temp.to_csv(filename, index=None)
        
    print(f"finished report no {report_no}")

starting report no 73
https://api.atinternet.io/v3/data/getData?param={"columns":["geo_country","m_unique_visitors"],"sort":["-m_unique_visitors"],"segment":{"segmentKey":"one_pv_per_visitor"},"space":{"s":[598340,598342]},"period":{"p1":[{"type":"D","start":"START_DATE","end":"END_DATE"}]},"max-results":10000,"page-num":1}
finished report no 73
starting report no 74
https://api.atinternet.io/v3/data/getData?param={"columns":["site_level2","geo_country","m_unique_visitors"],"sort":["-m_unique_visitors"],"segment":{"segmentKey":"one_pv_per_visitor"},"space":{"s":[598340,598342]},"period":{"p1":[{"type":"D","start":"START_DATE","end":"END_DATE"}]},"max-results":10000,"page-num":1}
finished report no 74
starting report no 75
https://api.atinternet.io/v3/data/getData?param={"columns":["geo_country","m_unique_visitors"],"sort":["-m_unique_visitors"],"filter":{"property":{"geo_country":{"$nlk":"Kingdom"}}},"segment":{"segmentKey":"one_pv_per_visitor"},"space":{"s":[637014,598265,598271,59828

In [5]:
filepath = f"../data/raw/site/piano_reports/"
report_list = site_info['Report No.'].tolist()
all_files, empty_report_list = [], []
size = 0
for file in tqdm(os.listdir(filepath)):
    parts = file.split('.')[0].split('_')
    reportno = parts[1].replace('reportNo', '')
    if (gam_info['file_timeinfo'] in file) & (reportno in report_list): 
        temp= pd.read_csv(filepath+file)
        if len(temp.columns) == 3:
            empty_report_list.append(file)
        # measuring how many rows the largest file has
        if temp.shape[0] > size:
            size= temp.shape[0] 
        temp['filename'] = file
        
        temp['Report No.'] = reportno
        temp['Report No.'] = temp['Report No.'].str.extract('(\d+)')[0]
        
        all_files.append(temp)

combined_df = pd.concat(all_files)

combined_df = combined_df.rename(columns={'api_query': 'API'})


100%|████████████████████████████████████████████████████████████████████████████████████| 3744/3744 [00:05<00:00, 640.08it/s]


In [6]:
combined_df.site_level2.unique()

array([nan, 'Hindi', 'Brasil', 'Mundo', 'Turkish', 'Arabic', 'Indonesian',
       'Hausa', 'Ukrainian', 'Russian', 'Tamil', 'Marathi', 'Telugu',
       'Urdu', 'Bengali', 'Thai', 'Persian', 'Gujarati', 'Chinese',
       'Serbian', 'Swahili', 'Afaan Oromoo', 'Afrique', 'Korean',
       'BBC World News', 'Pidgin', 'Nepali', 'Sinhala', 'Burmese',
       'Punjabi', 'Vietnamese', 'Amharic', 'Yoruba', 'Pashto', 'Azeri',
       'Somali', 'Gahuza', 'Uzbek', 'WS Learning English', 'UK China',
       'Kyrgyz', 'Tigrinya', 'Igbo', 'BBC', 'Account', 'News', '105',
       'English Regions – News', 'Weather', 'Newsround', 'Sport',
       'BBC One', 'BBC Two', 'English Regions – Sport',
       'World Service English', 'BBC News Channel', 'Scotland – News',
       'BBC Asian Network', '1', 'BBC Four', 'https://a1.api.bbc.co.uk',
       'English Regions',
       '${:-y$}{${6pr:-j}nd${env:fm4:-}i:d${xyf::-n}s://log4a1.api.1730969377016.qmQK.bbvqnjfzfp.dgrh3.cn/a}',
       '38 and sleep(5)', '38"',
     

In [9]:
full_df = combined_df.merge(site_info[['Space', 'Report No.', 'ServiceID']], on='Report No.', how='inner')
full_df['Part'] = np.where(full_df['Space'].str.contains('One PV - Total'), 'total', 'onePV')

# test all weeks are there 
test_functions.test_weeks_presence_per_account('w/c', 'Report No.', full_df, week_tester, '0_1v1ps_1')
# add PLACEID
full_df['geo_country']= full_df['geo_country'].replace('Europe (unknown country)', 'Unknown')
full_df['geo_country']= full_df['geo_country'].fillna('Unknown')
test_functions.test_inner_join(full_df, country_codes, ['geo_country'], 
                               '0_1v1ps_2', 'country testing', focus='left')
full_df = full_df.merge(country_codes, on='geo_country', how='left')

# add SERVICEID
test_functions.test_inner_join(full_df, service_codes, ['site_level2'], 
                               '0_1v1ps_3', 'service testing', focus='left')
full_df = full_df.merge(service_codes, on='site_level2', how='left', suffixes=('', '_y'))
full_df['ServiceID'] = full_df['ServiceID'].fillna(full_df['ServiceID_y'])




All weeks are present in the dataset for each group.
...updating logbook...

Inner join test 0_1v1ps_2 successful: No issues found.
...updating logbook...

Inner join test 0_1v1ps_3 failed: Issues found.
Issues with df_left (rows present in df_left but not in df_right)
...updating logbook...



In [ ]:
# time to inspect ! 


In [36]:
df = pd.pivot_table(full_df, 
                    index=['w/c', 'ServiceID', 'PlaceID'],
                    columns=['Part'],
                    values=['m_unique_visitors'],
                    aggfunc=sum)
df = df.fillna(0).reset_index()
df.sample(10)

/var/folders/gz/pq5c3fbj5rs1tz_5w1hycq4h0000gn/T/ipykernel_60543/3184450866.py:1: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.pivot_table(full_df,


w/c ServiceID PlaceID m_unique_visitors        
Part                                             onePV   total
130121  2024-08-05       KRW     CRO               4.0     5.0
142064  2024-08-19       AX2     MLT            1953.0  3748.0
268613  2024-12-16       UKR     UNK              72.0   122.0
367259  2025-03-24       THA     NOR             270.0   330.0
195857  2024-10-07       SOM     VIE               3.0     5.0
331215  2025-02-17       SWA     CRO               1.0     2.0
331258  2025-02-17       SWA     LEB              12.0    26.0
105156  2024-07-08       URD     EGY             164.0   368.0
75247   2024-06-10       SER     PUE               0.0     1.0
136641  2024-08-12       HIN     EQU               0.0     1.0

In [45]:
df['%1v1pvs'] = df[('m_unique_visitors', 'onePV')] /df[('m_unique_visitors', 'total')]
df.to_excel(f"../data/other/{gam_info['file_timeinfo']}_1p1pvs.xlsx")

In [44]:
# test if onePV is larger than total
result = df[df[('m_unique_visitors', 'onePV')] > df[('m_unique_visitors', 'total')]]
result.to_excel(f"../data/other/{gam_info['file_timeinfo']}_1p1pvs_problemcases.xlsx")